# 4. SQL sorguları

`analytics/sql/` klasöründeki sorgular raporların ve sitenin altyapısı. Kullanılan teknikler: CTE, pencere fonksiyonları (`ROW_NUMBER`, `RANK`, `PERCENT_RANK`, `LAG`, kayan `AVG`/`MAX`), koşullu toplama ile pivot, alt sorgular.

In [1]:
import sys, sqlite3
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from mi import db, stats as st, events as ev, charts
%matplotlib inline
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
con = db.connect()
prices = pd.read_sql('SELECT * FROM prices', con)
close, volume = st.wide(prices), st.wide(prices, 'volume')
r = st.returns(close)
assets = pd.read_sql('SELECT * FROM assets', con).set_index('code')
print(f'{close.shape[1]} varlık, {close.index[0]:%Y-%m-%d} – {close.index[-1]:%Y-%m-%d}, {len(prices):,} fiyat satırı')

109 varlık, 2021-10-11 – 2026-10-09, 130,517 fiyat satırı


In [2]:
from IPython.display import Markdown
for f in sorted(db.SQL_DIR.glob('*.sql')):
    sql = f.read_text()
    display(Markdown(f'### {f.stem}\n```sql\n{sql}\n```'))
    display(db.query(f.stem, con).head(8))

### drawdowns
```sql
-- Her varlığın zirveden en büyük düşüşü (maksimum drawdown) ve şu an zirveden ne kadar uzakta olduğu.
-- Teknik: MAX() OVER (ROWS UNBOUNDED PRECEDING) ile o güne kadarki en yüksek kapanış.
WITH peak AS (
  SELECT code, date, close,
         MAX(close) OVER (PARTITION BY code ORDER BY date ROWS UNBOUNDED PRECEDING) AS run_max,
         ROW_NUMBER() OVER (PARTITION BY code ORDER BY date DESC) AS rn
  FROM prices
),
dd AS (SELECT code, date, rn, 100.0 * (close / run_max - 1) AS drawdown FROM peak)
SELECT d.code, a.name, a.cls,
       ROUND(MIN(drawdown), 1)                                AS max_drawdown,
       (SELECT date FROM dd x WHERE x.code = d.code ORDER BY drawdown LIMIT 1) AS trough_date,
       ROUND(MAX(CASE WHEN rn = 1 THEN drawdown END), 1)      AS current_drawdown
FROM dd d JOIN assets a USING (code)
GROUP BY d.code
ORDER BY max_drawdown;

```

,code,name,cls,max_drawdown,trough_date,current_drawdown
0,MRNA,Moderna,hisse,-93.90,2025-11-20,-38.90
1,APP,AppLovin,hisse,-91.90,2022-12-27,-62.20
2,PYPL,PayPal Holdings,hisse,-85.60,2026-02-12,-79.30
3,SHOP,Shopify,hisse,-84.80,2022-10-11,-4.60
4,MSTR,Strategy,hisse,-84.10,2022-12-29,-67.40
5,DASH,DoorDash,hisse,-82.50,2022-10-14,-30.30
6,RKLB,Rocket Lab,hisse,-78.30,2024-04-15,-54.60
7,PLTR,Palantir Technologies,hisse,-77.60,2022-12-27,0.00


### fund_consensus
```sql
-- Takip edilen büyük fonların ortak tercihleri: kaç fon tutuyor, son çeyrekte kaçı artırdı/azalttı.
SELECT code,
       COUNT(*)                                                     AS funds,
       SUM(CASE WHEN change IN ('new', 'add') THEN 1 ELSE 0 END)    AS buyers,
       SUM(CASE WHEN change IN ('cut', 'out') THEN 1 ELSE 0 END)    AS sellers,
       ROUND(SUM(value_usd) / 1e9, 2)                               AS total_busd
FROM fund_holdings
WHERE code IS NOT NULL
GROUP BY code
HAVING funds >= 3
ORDER BY buyers - sellers DESC, funds DESC;

```

,code,funds,buyers,sellers,total_busd
0,GOOGL,24,16,6,48.10
1,AMZN,12,8,4,12.72
2,WBD,3,3,0,0.74
3,AMD,8,5,3,2.23
4,INTC,5,3,1,3.32
5,LIN,3,2,0,0.97
6,ISRG,3,2,1,0.54
7,AVGO,9,4,4,3.96


### insider_by_sector
```sql
-- Son 180 günde yöneticilerin sektör bazında net alım/satımı ve plansız (10b5-1 dışı) satışların payı.
SELECT a.sector,
       COUNT(DISTINCT t.code)                                   AS companies,
       ROUND(SUM(t.buy_usd) / 1e6, 1)                           AS buy_musd,
       ROUND(SUM(t.sell_usd) / 1e6, 1)                          AS sell_musd,
       ROUND((SUM(t.buy_usd) - SUM(t.sell_usd)) / 1e6, 1)       AS net_musd,
       ROUND(100.0 * SUM(CASE WHEN t.plan = 0 THEN t.sell_usd ELSE 0 END) / NULLIF(SUM(t.sell_usd), 0), 0) AS unplanned_sell_pct
FROM insider_trades t JOIN assets a USING (code)
WHERE t.date >= date((SELECT MAX(date) FROM prices), '-180 day')
GROUP BY a.sector
ORDER BY net_musd DESC;

```

...

,code,period_end,revenue,yoy,acceleration
0,AAPL,2024-09-28,"94,930,000,000.00",6.07,NaN
1,AAPL,2024-12-28,"124,300,000,000.00",3.95,-2.12
2,AAPL,2025-03-29,"95,359,000,000.00",5.08,1.12
3,AAPL,2025-06-28,"94,036,000,000.00",9.63,4.55
4,AAPL,2025-09-27,"102,466,000,000.00",7.94,-1.69
5,AAPL,2025-12-27,"143,756,000,000.00",15.65,7.71
6,AAPL,2026-03-28,"111,184,000,000.00",16.60,0.94
7,AAPL,2026-06-27,"109,417,000,000.00",16.36,-0.24


### sector_returns
```sql
-- Sektörlerin eşit ağırlıklı getirisi ve sektör içi en iyi / en kötü hisse.
-- Teknik: CTE zinciri, pencere fonksiyonu ile sektör içi sıralama (RANK), GROUP BY ile toplulaştırma.
WITH ranked AS (
  SELECT code, date, close, ROW_NUMBER() OVER (PARTITION BY code ORDER BY date DESC) AS rn
  FROM prices
),
ret AS (
  SELECT code,
         100.0 * (MAX(CASE WHEN rn = 1 THEN close END) / MAX(CASE WHEN rn = 6 THEN close END) - 1)  AS r_1w,
         100.0 * (MAX(CASE WHEN rn = 1 THEN close END) / MAX(CASE WHEN rn = 22 THEN close END) - 1) AS r_1m,
         100.0 * (MAX(CASE WHEN rn = 1 THEN close END) / MAX(CASE WHEN rn = 64 THEN close END) - 1) AS r_3m
  FROM ranked WHERE rn <= 64 GROUP BY code
),
stock AS (
  SELECT a.sector, r.*,
         RANK() OVER (PARTITION BY a.sector ORDER BY r.r_1m DESC) AS best,
         RANK() OVER (PARTITION BY a.sector ORDER BY r.r_1m ASC)  AS worst
  FROM ret r JOIN assets a USING (code)
  WHERE a.cls = 'hisse' AND a.sector <> ''
)
SELECT sector,
       COUNT(*)                                  AS n,
       ROUND(AVG(r_1w), 2)                       AS r_1w,
       ROUND(AVG(r_1m), 2)                       AS r_1m,
       ROUND(AVG(r_3m), 2)                       AS r_3m,
       SUM(CASE WHEN r_1m > 0 THEN 1 ELSE 0 END) AS up_1m,
       MAX(CASE WHEN best = 1 THEN code END)     AS leader,
       MAX(CASE WHEN worst = 1 THEN code END)    AS laggard
FROM stock
GROUP BY sector
ORDER BY r_1m DESC;

```

,sector,n,r_1w,r_1m,r_3m,up_1m,leader,laggard
0,Yarı iletken,18,-5.20,9.51,0.98,18,MRVL,AVGO
1,Teknoloji,26,1.83,8.83,13.16,16,SHOP,WDC
2,Sağlık,10,4.32,8.71,27.38,6,MRNA,ALNY
3,Havacılık ve uzay,4,-1.56,6.32,-6.60,4,RKLB,HONA
4,Tüketici hizmetleri,2,4.53,0.96,10.24,1,PYPL,DASH
5,Seyahat ve eğlence,3,1.63,0.77,1.94,1,MAR,BKNG
6,Perakende,10,2.86,0.17,-0.33,6,WMT,CPRT
7,Gıda ve içecek,5,2.65,-1.19,-3.12,3,KDP,PEP


### trend_status
```sql
-- 50 ve 200 günlük hareketli ortalamalar ve fiyatın bunlara göre konumu (trend sağlığı).
-- Teknik: AVG() OVER (ROWS BETWEEN n PRECEDING AND CURRENT ROW) ile hareketli ortalama.
WITH ma AS (
  SELECT code, date, close,
         AVG(close) OVER (PARTITION BY code ORDER BY date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW)  AS ma50,
         AVG(close) OVER (PARTITION BY code ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS ma200,
         COUNT(*)   OVER (PARTITION BY code ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS n,
         ROW_NUMBER() OVER (PARTITION BY code ORDER BY date DESC) AS rn
  FROM prices
)
SELECT code, date, close, ROUND(ma50, 2) AS ma50, ROUND(ma200, 2) AS ma200,
       ROUND(100.0 * (close / ma200 - 1), 1) AS dist_ma200,
       CASE WHEN close > ma50 AND ma50 > ma200 THEN 'güçlü yükseliş'
            WHEN close > ma200                  THEN 'yükseliş'
            WHEN close < ma50 AND ma50 < ma200  THEN 'güçlü düşüş'
            ELSE 'düşüş' END AS trend
FROM ma
WHERE rn = 1 AND n = 200;

```

,code,date,close,ma50,ma200,dist_ma200,trend
0,AAPL,2026-10-09,336.64,322.36,290.40,15.90,güçlü yükseliş
1,ABNB,2026-10-09,165.87,171.04,144.20,15.00,yükseliş
2,ADBE,2026-10-09,242.27,257.88,256.98,-5.70,düşüş
3,ADI,2026-10-09,406.19,379.62,361.30,12.40,güçlü yükseliş
4,ADP,2026-10-09,270.88,270.77,235.25,15.10,güçlü yükseliş
5,ADSK,2026-10-09,235.35,233.48,237.32,-0.80,düşüş
6,AEP,2026-10-09,122.81,122.72,125.90,-2.50,düşüş
7,ALAB,2026-10-09,342.21,317.46,246.36,38.90,güçlü yükseliş
